# RTB ≡ Trust-PCL: A Practical Demonstration

**Deleu et al. (2025)** proved that Relative Trajectory Balance (RTB) — a GFlowNet
training objective — is mathematically equivalent to Trust-PCL, an off-policy RL
method with KL regularization ([arXiv:2509.01632](https://arxiv.org/abs/2509.01632)).

This notebook demonstrates the equivalence empirically: we train **two completely
independent GFlowNets** — one using RTB, one using Trust-PCL — from cloned
initializations, and show that the **loss relationship holds exactly** at every step.

### Parameter correspondence

| RTB (GFlowNet) | Trust-PCL (RL) | Relationship |
|:---|:---|:---|
| `beta` (reward scaling) | `alpha` (temperature) | `alpha = 1/beta` |
| `logZ` (log partition) | `v_soft_s0` (soft value at s₀) | `v_soft_s0 = alpha * logZ` |
| `pf` (posterior policy) | `policy` | Same role |
| `prior_pf` (prior policy) | `reference_policy` | Same role |
| `L_RTB` | `L_Trust-PCL` | `L_TPCL = α² · L_RTB` |

### The α² scaling and optimizer choice

Since `L_TPCL = α² · L_RTB`, the gradients differ by a factor of α². With a
learning-rate-compensated **SGD** optimizer (`lr_tpcl = lr_rtb / α²`), the
parameter updates are bitwise identical — but SGD converges poorly on this
joint optimization problem (logZ + policy have very different curvatures).

We therefore use **Adam**, which converges well. The loss relationship
`L_TPCL = α² · L_RTB` still holds exactly at every step (it's a mathematical
identity, independent of the optimizer). However, Adam's adaptive step size
normalizes by `sqrt(v) + ε`, and since `grad_tpcl = α² · grad_rtb`, the
second moment `v` differs by `α⁴`. The `ε` term prevents exact cancellation,
causing a small weight divergence over training.

We use `beta=2.0` / `alpha=0.5` so α²=0.25 and the scaling is clearly visible.

In [7]:
import matplotlib.pyplot as plt
import torch

from gfn.estimators import DiscretePolicyEstimator
from gfn.gflownet import RelativeTrajectoryBalanceGFlowNet, TrustPCLGFlowNet
from gfn.gym import HyperGrid
from gfn.preprocessors import KHotPreprocessor
from gfn.samplers import Sampler
from gfn.utils.common import set_seed
from gfn.utils.modules import DiscreteUniform, MLP
from gfn.utils.training import validate

## 1. Environment and shared hyperparameters

We use a small 2D HyperGrid (8×8 = 64 states) with `calculate_partition=True` so we can compare the learned distribution to the ground truth.

In [ ]:
SEED = 42
BETA = 2.0                     # RTB reward scaling
ALPHA = 1.0 / BETA             # Trust-PCL temperature = 0.5
LR_RTB = 1e-3                  # RTB learning rate
LR_TPCL = LR_RTB / ALPHA**2   # Compensated: lr / α² = 1e-3 / 0.25 = 4e-3
N_ITERATIONS = 3000
BATCH_SIZE = 32
HIDDEN_DIM = 64

print(f"beta={BETA}, alpha={ALPHA}, alpha^2={ALPHA**2}")
print(f"lr_rtb={LR_RTB}, lr_tpcl={LR_TPCL}")

# Environment (shared — both GFlowNets train on the same env).
env = HyperGrid(
    ndim=2,
    height=8,
    calculate_partition=True,
    store_all_states=True,
)
preprocessor = KHotPreprocessor(height=env.height, ndim=env.ndim)
print(f"\nHyperGrid: {env.height}^{env.ndim} = {env.n_states} states")

## 2. Build two independent GFlowNets from cloned weights

We create the RTB GFlowNet first, then build the Trust-PCL GFlowNet with
**separate** neural networks whose weights are cloned from RTB. This ensures
they start identical but are trained independently — no shared state.

### About the prior (reference policy) and what logZ estimates

RTB learns a posterior `p_post(x) ∝ p_prior(x) · r(x)^β`. The learned `logZ`
converges to `log Σ_x p_prior(x) · r(x)^β`, which depends on the prior.

We use a **uniform prior** (equal probability for all *actions* at every state).
Crucially, uniform over actions ≠ uniform over states: the exit action fires
at every step, so short trajectories are far more likely. The origin gets ~33%
of prior mass while the far corner gets ~0.2%.

This means `logZ` converges to a value much smaller than the environment's raw
`log Σ r(x)` — because the prior downweights states far from the origin. This
is correct: `logZ` estimates the normalizing constant of the *prior-tilted*
target, not the raw reward.

**The RTB ↔ Trust-PCL equivalence holds regardless of the prior choice.**

In [19]:
def make_posterior(preprocessor, n_actions, hidden_dim):
    """Build a trainable DiscretePolicyEstimator with an MLP."""
    module = MLP(
        input_dim=preprocessor.output_dim,
        output_dim=n_actions,
        hidden_dim=hidden_dim,
    )
    return DiscretePolicyEstimator(
        module=module,
        n_actions=n_actions,
        preprocessor=preprocessor,
        is_backward=False,
    )


def make_uniform_prior(preprocessor, n_actions):
    """Build a fixed uniform prior (no learnable parameters)."""
    return DiscretePolicyEstimator(
        module=DiscreteUniform(n_actions),
        n_actions=n_actions,
        preprocessor=preprocessor,
        is_backward=False,
    )


# --- RTB GFlowNet ---
set_seed(SEED)
pf_rtb = make_posterior(preprocessor, env.n_actions, HIDDEN_DIM)
prior_rtb = make_uniform_prior(preprocessor, env.n_actions)
gfn_rtb = RelativeTrajectoryBalanceGFlowNet(
    pf=pf_rtb, prior_pf=prior_rtb, beta=BETA, init_logZ=0.0,
)

# --- Trust-PCL GFlowNet (cloned posterior, same uniform prior) ---
pf_tpcl = make_posterior(preprocessor, env.n_actions, HIDDEN_DIM)
prior_tpcl = make_uniform_prior(preprocessor, env.n_actions)
# Clone posterior weights so both start identical.
pf_tpcl.load_state_dict(pf_rtb.state_dict())

gfn_tpcl = TrustPCLGFlowNet(
    policy=pf_tpcl,
    reference_policy=prior_tpcl,
    alpha=ALPHA,
    init_v_soft_s0=ALPHA * 0.0,  # = alpha * init_logZ = 0.0
)

# Verify: initial posterior weights are identical.
for (n1, p1), (n2, p2) in zip(
    pf_rtb.named_parameters(), pf_tpcl.named_parameters()
):
    assert torch.equal(p1, p2), f"Mismatch in {n1}"
print("Initial posterior weights are identical between RTB and Trust-PCL.")
print(f"Prior: uniform (DiscreteUniform, no learnable parameters)")

Initial posterior weights are identical between RTB and Trust-PCL.
Prior: uniform (DiscreteUniform, no learnable parameters)


## 3. Verify the exact loss identity (before training)

With identical weights and the same trajectories, we can verify that
`L_TPCL = α² · L_RTB` holds exactly. This is the core mathematical result.

In [ ]:
# Before any training: weights are identical, so we can compare directly.
torch.manual_seed(0)
sampler_rtb = Sampler(estimator=pf_rtb)
trajs = sampler_rtb.sample_trajectories(env, n=64, save_logprobs=True)

with torch.no_grad():
    loss_rtb = gfn_rtb.loss(env, trajs, recalculate_all_logprobs=True)
    loss_tpcl = gfn_tpcl.loss(env, trajs, recalculate_all_logprobs=True)

print(f"RTB loss:              {loss_rtb.item():.6f}")
print(f"Trust-PCL loss:        {loss_tpcl.item():.6f}")
print(f"alpha^2 * RTB loss:    {ALPHA**2 * loss_rtb.item():.6f}")
print(f"Exact match:           {torch.allclose(loss_tpcl, ALPHA**2 * loss_rtb, atol=1e-6)}")
print(f"\nRatio L_TPCL / L_RTB = {loss_tpcl.item() / loss_rtb.item():.6f} (should be alpha^2 = {ALPHA**2})")

## 4. Train both independently with Adam

Now we train both GFlowNets independently. Since Adam's adaptive step size
interacts with the α² gradient scaling (see discussion below), the weights
will diverge slightly over training. But both should converge to similar
solutions — similar loss values and similar logZ estimates.

In [ ]:
# Separate Adam optimizers with compensated learning rates.
opt_rtb = torch.optim.Adam(
    list(gfn_rtb.pf.parameters()) + gfn_rtb.logz_parameters(), lr=LR_RTB,
)
opt_tpcl = torch.optim.Adam(
    list(gfn_tpcl.pf.parameters()) + gfn_tpcl.logz_parameters(), lr=LR_TPCL,
)

# Separate samplers (each uses its own posterior policy).
sampler_tpcl = Sampler(estimator=pf_tpcl)

# Logging.
log = {
    "loss_rtb": [], "loss_tpcl": [],
    "logZ_rtb": [], "logZ_tpcl": [],
    "weight_diff": [],
}

for step in range(N_ITERATIONS):
    # --- RTB step ---
    torch.manual_seed(SEED + step)
    trajs_rtb = sampler_rtb.sample_trajectories(env, n=BATCH_SIZE, save_logprobs=True)
    opt_rtb.zero_grad()
    loss_rtb = gfn_rtb.loss(env, trajs_rtb)
    loss_rtb.backward()
    opt_rtb.step()

    # --- Trust-PCL step ---
    torch.manual_seed(SEED + step)
    trajs_tpcl = sampler_tpcl.sample_trajectories(env, n=BATCH_SIZE, save_logprobs=True)
    opt_tpcl.zero_grad()
    loss_tpcl = gfn_tpcl.loss(env, trajs_tpcl)
    loss_tpcl.backward()
    opt_tpcl.step()

    # --- Log ---
    with torch.no_grad():
        weight_diff = sum(
            (p1 - p2).abs().sum().item()
            for p1, p2 in zip(pf_rtb.parameters(), pf_tpcl.parameters())
        )

    log["loss_rtb"].append(loss_rtb.item())
    log["loss_tpcl"].append(loss_tpcl.item())
    log["logZ_rtb"].append(gfn_rtb.logZ.item())
    log["logZ_tpcl"].append(gfn_tpcl.v_soft_s0.item() / ALPHA)
    log["weight_diff"].append(weight_diff)

print(f"Final logZ (RTB):              {log['logZ_rtb'][-1]:.4f}")
print(f"Final v_soft_s0/alpha (TPCL):  {log['logZ_tpcl'][-1]:.4f}")
print(f"Final weight difference:       {log['weight_diff'][-1]:.4f}")

## 5. Results

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12))

# --- Panel 1: Loss curves (both models independently) ---
ax = axes[0]
ax.plot(log["loss_rtb"], label="RTB loss", alpha=0.5, linewidth=2)
ax.plot(log["loss_tpcl"], label="Trust-PCL loss", linestyle="--", alpha=0.5)
ax.set_xlabel("Training step")
ax.set_ylabel("Loss")
ax.set_title("Both models converge (trained independently with Adam)")
ax.legend()
ax.set_yscale("log")

# --- Panel 2: logZ convergence ---
ax = axes[1]
ax.plot(log["logZ_rtb"], label="logZ (RTB)", alpha=0.5, linewidth=2)
ax.plot(log["logZ_tpcl"], label="v_soft_s0 / α (Trust-PCL)",
        linestyle="--", alpha=0.5)
ax.set_xlabel("Training step")
ax.set_ylabel("Value")
ax.set_title("Both learn the same partition function (slight divergence from Adam)")
ax.legend()

# --- Panel 3: Weight difference ---
ax = axes[2]
ax.plot(log["weight_diff"])
ax.set_xlabel("Training step")
ax.set_ylabel("Sum of |weight differences|")
ax.set_title("Weight divergence (Adam artifact — would be 0 with SGD)")

plt.tight_layout()
plt.show()

## 6. What this demonstrates

**Section 3** showed the exact mathematical identity: with the same parameters
and the same trajectories, `L_TPCL = α² · L_RTB`. This is a tautology — it
follows directly from `TrustPCLGFlowNet.loss()` returning `α² * super().loss()`.

**Sections 4-5** showed the practical consequence: two independently trained
models (one RTB, one Trust-PCL) with Adam converge to similar solutions. The
weight divergence in the bottom panel is an optimizer artifact, not a violation
of the equivalence.

### Why the weights diverge with Adam

The loss identity `L_TPCL = α² · L_RTB` means the gradients satisfy
`∇L_TPCL = α² · ∇L_RTB`. With plain **SGD** and `lr_tpcl = lr_rtb / α²`,
the updates would be identical: `lr · grad = (lr/α²) · (α² · grad)`.

But **Adam** normalizes each parameter's gradient by its running second
moment: `update = lr · m / (√v + ε)`. Since `grad_tpcl = α² · grad_rtb`,
the second moments differ by `α⁴`, and the small `ε` term (default 1e-8)
prevents exact cancellation. The result is a tiny per-step difference that
accumulates over training.

SGD produces zero weight divergence but converges poorly on this problem
because logZ and policy parameters have very different curvatures — Adam's
adaptive step sizes are essential for practical GFlowNet training.

### The takeaway

RTB (GFlowNet) and Trust-PCL (RL) are **mathematically identical** —
they define the same loss surface over the same parameters, differing
only by a constant factor α². The only difference is vocabulary:

- A GFlowNet researcher says "posterior policy", "prior", "logZ", "beta"
- An RL researcher says "policy", "reference policy", "soft value", "alpha"

This equivalence (Deleu et al. 2025, [arXiv:2509.01632](https://arxiv.org/abs/2509.01632))
means that insights from entropy-regularized RL (soft actor-critic, Trust-PCL,
KL-regularized policy optimization) transfer directly to GFlowNet training,
and vice versa.